El notebook tendrá 6 secciones clave

El notebook completo queda así:
```sh
sagemaker_byoc.ipynb
├── Celda 1 — Setup e imports
├── Celda 2 — Build y push a ECR
├── Celda 3 — Training Job
├── Celda 4 — Desplegar Endpoint
├── Celda 5 — Inferencias en tiempo real
└── Celda 6 — Limpieza de recursos
```

In [1]:
# Sección 1 - setup e imports 
# Esta celda configura todo lo necesario

import boto3
import sagemaker
from sagemaker import get_execution_role

# Sesión de SageMaker
sess = sagemaker.Session()
role = get_execution_role()

# Datos de la cuenta
account_id = boto3.client("sts").get_caller_identity()["Account"]
region = boto3.session.Session().region_name

# Nombres
bucket = "ml-predsales-bucket"
image_name = "ml-predsales-training"
prefix = "data/training"

print(f"Account: {account_id}")
print(f"Region:  {region}")
print(f"Role:    {role}")
print(f"Image:   {account_id}.dkr.ecr.{region}.amazonaws.com/{image_name}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Account: 561137843164
Region:  us-east-1
Role:    arn:aws:iam::561137843164:role/SageMakerStudioExecutionRole2026
Image:   561137843164.dkr.ecr.us-east-1.amazonaws.com/ml-predsales-training


In [7]:
# Sección 2 - Build y push de la imagen a ECR 
# En esta celda vamos a construir la imagen docker y subirla a ECR

import os
import subprocess

# Ruta al Dockerfile
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
training_dir = f"{repo_root}/src/training"

# URI completa de la imagen en ECR
ecr_image = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{image_name}:latest"

# Crear el repositorio en ECR si no existe
os.system(f"aws ecr describe-repositories --repository-names {image_name} > /dev/null 2>&1 || aws ecr create-repository --repository-name {image_name} > /dev/null")

# Login a ECR
os.system(f"aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com")

# Build con --network sagemaker (requerido en SageMaker Studio)
os.system(f"docker build --network sagemaker -t {image_name} {training_dir}")

# Tagear y pushear
os.system(f"docker tag {image_name}:latest {ecr_image}")
os.system(f"docker push {ecr_image}")

print(f"Image pushed: {ecr_image}")


WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.



Login Succeeded
Sending build context to Docker daemon  43.52kB
Step 1/13 : FROM python:3.12-slim
 ---> 6f90d4a79e7a
Step 2/13 : ENV PATH="/opt/program:${PATH}"
 ---> Using cache
 ---> 8f59b6a69aa1
Step 3/13 : WORKDIR /opt/program
 ---> Using cache
 ---> 66de42a158d2
Step 4/13 : COPY requirements.txt .
 ---> Using cache
 ---> 9b84ca6a8918
Step 5/13 : RUN pip install --no-cache-dir -r requirements.txt
 ---> Using cache
 ---> bc4394973697
Step 6/13 : COPY train.py .
 ---> Using cache
 ---> c651d87a5b55
Step 7/13 : COPY predictor.py .
 ---> Using cache
 ---> caca4ade852f
Step 8/13 : COPY utils/ utils/
 ---> Using cache
 ---> 7539c2bfbc08
Step 9/13 : COPY __main__.py .
 ---> Using cache
 ---> 660cd2d4a9ec
Step 10/13 : COPY train train
 ---> 4446d4eb402d
Step 11/13 : COPY serve serve
 ---> dff47f651a8b
Step 12/13 : RUN chmod +x train serve
 ---> Running in 7d155fbcf3ab
 ---> Removed intermediate container 7d155fbcf3ab
 ---> 131e445aeaa6
Step 13/13 : LABEL com.amazon.studio.user.resources=tr

### ¿Qué hace cada paso?
```sh
docker build    → construye la imagen desde tu Dockerfile
ecr create      → crea el repositorio en ECR (solo la primera vez)
ecr get-login   → se autentica con ECR
docker tag      → le pone la URI completa de ECR a la imagen
docker push     → sube la imagen a ECR
```

El resultado final es tu imagen disponible en:
561137843164.dkr.ecr.us-east-1.amazonaws.com/ml-predsales-training:latest


In [8]:
# Sección 3 - Training Job 
#En esta celda se lanza el entrenamiento en SM 

from sagemaker.estimator import Estimator

# URI de la imagen en ECR
ecr_image = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{image_name}:latest"

# Definir el Estimator
estimator = Estimator(
    image_uri=ecr_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size=10,
    max_run=7200,
    output_path=f"s3://{bucket}/output",
    sagemaker_session=sess,
    hyperparameters={
        "n_estimators": 100,
        "max_depth": 10,
        "random_seed": 10,
        "use_random_search": "false",
    }
)

# Lanzar el training job
estimator.fit(
    {"training": f"s3://{bucket}/{prefix}"},
    job_name="ml-predsales-training-job-v2",
    wait=True,
    logs=True,
)

print("Training job complete!")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: ml-predsales-training-job-v2


2026-03-08 21:25:24 Starting - Starting the training job...
2026-03-08 21:25:38 Starting - Preparing the instances for training...
2026-03-08 21:26:00 Downloading - Downloading input data..
2026-03-08 21:26:30 Training - Training image download completed. Training in progress.2026-03-08 21:26:35,788 - train - INFO - ==================================================
2026-03-08 21:26:35,788 - train - INFO - Starting training pipeline (train.py)
2026-03-08 21:26:35,788 - train - INFO - ==================================================
2026-03-08 21:26:35,788 - train - INFO - Loading prepared dataset...
2026-03-08 21:26:39,754 - train - INFO - Prepared dataset loaded: 9,330,156 records
2026-03-08 21:26:40,941 - train - INFO - Train set: 8,906,058 records (months 0–32)
2026-03-08 21:26:40,941 - train - INFO - Validation set: 424,098 records (month 33)
2026-03-08 21:26:40,981 - train - INFO - Training Random Forest — n_estimators=100, max_depth=10, random_state=10
2026-03-08 21:31:03,147 -

In [11]:
# Sección pre-4 
# Por intentos del training, tengo que eliminar la configuración del endpoint 
import boto3

sm = boto3.client("sagemaker")

# Eliminar endpoint config si existe
try:
    sm.delete_endpoint_config(EndpointConfigName="ml-predsales-endpoint")
    print("Endpoint config eliminada")
except:
    print("No existía endpoint config, continuamos")

# Eliminar endpoint si existe
try:
    sm.delete_endpoint(EndpointName="ml-predsales-endpoint")
    print("Endpoint eliminado")
except:
    print("No existía endpoint, continuamos")

Endpoint config eliminada
Endpoint eliminado


In [12]:
# Sección 4 - desplegar el endpoint 
#Esta celda despliega el modelo como un endpoint de inferencia en tiempo real 

from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

# Desplegar el endpoint
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="ml-predsales-endpoint",
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer(),
)

print(f"Endpoint deployed: {predictor.endpoint_name}")


INFO:sagemaker:Creating model with name: ml-predsales-training-2026-03-08-21-43-36-520
INFO:sagemaker:Creating endpoint-config with name ml-predsales-endpoint
INFO:sagemaker:Creating endpoint with name ml-predsales-endpoint


----!Endpoint deployed: ml-predsales-endpoint


### ¿Qué hace cada parte?
**`estimator.deploy()`** hace tres cosas automáticamente:
```sh
1. Toma el modelo que guarde en S3 tras el training job
2. Lanza una instancia `ml.m5.large` con tu imagen Docker
3. Corre tu script `serve` que levanta gunicorn con Flask
```

- **`CSVSerializer`** convierte tus datos a CSV antes de mandarlos al endpoint
- **`CSVDeserializer`** convierte la respuesta CSV del endpoint de vuelta a Python

El resultado es un endpoint disponible en:
https://runtime.sagemaker.us-east-1.amazonaws.com/endpoints/ml-predsales-endpoint/invocations

In [13]:
# Sección 5 - Inferencias en Tiempo Real 
# En esta celda se generan predicciones usando el endpoint 

import numpy as np
import pandas as pd

# Muestra de datos para inferencia
# Columnas: lag_1, lag_3, lag_6, lag_12
sample_data = pd.DataFrame([
    [5.0, 4.0, 3.0, 2.0],
    [3.0, 2.0, 1.0, 0.5],
    [10.0, 8.0, 6.0, 4.0],
    [0.0, 0.0, 0.0, 0.0],
    [7.0, 6.0, 5.0, 3.0],
], columns=["lag_1", "lag_3", "lag_6", "lag_12"])

# Hacer predicciones
response = predictor.predict(sample_data.values)

# Mostrar resultados
predictions = [float(x[0]) for x in response]
results_df = sample_data.copy()
results_df["predicted_sales"] = predictions

print("Inferencias en tiempo real:")
print(results_df.to_string(index=False))



Inferencias en tiempo real:
 lag_1  lag_3  lag_6  lag_12  predicted_sales
   5.0    4.0    3.0     2.0         3.165871
   3.0    2.0    1.0     0.5         1.436864
  10.0    8.0    6.0     4.0         6.120681
   0.0    0.0    0.0     0.0         0.077701
   7.0    6.0    5.0     3.0         5.000904


### ¿Qué pasa aquí?
```sh
sample_data (DataFrame)
      ↓ CSVSerializer lo convierte a CSV
POST /invocations en el endpoint
      ↓ predictor.py corre model.predict()
      ↓ CSVDeserializer convierte respuesta
predictions (lista de floats)
```
El resultado se ve así:
```
lag_1  lag_3  lag_6  lag_12  predicted_sales
  5.0    4.0    3.0     2.0              4.2
  3.0    2.0    1.0     0.5              1.8
 10.0    8.0    6.0     4.0              8.5
  0.0    0.0    0.0     0.0              0.0
  7.0    6.0    5.0     3.0              5.9
```


In [16]:
# Sección 6 - Limpieza de recursos
# Esta celda elimina el endpoint para no generar costos innecesarios

import boto3

sm = boto3.client("sagemaker")

try:
    sm.delete_endpoint(EndpointName="ml-predsales-endpoint")
    print("Endpoint deleted: ml-predsales-endpoint")
except Exception as e:
    print(f"Endpoint ya no existe o error: {e}")

Endpoint ya no existe o error: An error occurred (ValidationException) when calling the DeleteEndpoint operation: Could not find endpoint "ml-predsales-endpoint".


### ¿Por qué es importante esta celda?

Un endpoint en SageMaker cobra por hora mientras esté corriendo, aunque se esté usando. 
